# 03 – Analyse de sentiment avec Spark MLlib

In [6]:
import time
from pyspark.sql import functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF
from pyspark.ml.classification import LogisticRegression, NaiveBayes
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from shopstream_utils import get_spark, read_table, write_table, MODEL_DIR

spark = get_spark("03-ml")

Spark 3.5.3 prêt - Spark UI : http://localhost:4040


## 4.1 Données étiquetées

Les avis sont transformés en données étiquetées pour l'apprentissage supervisé : les notes de 4 ou 5 deviennent la classe positive (`label = 1`), celles de 1 ou 2 la classe négative (`label = 0`). Les avis à 3 étoiles sont exclus, puis les données sont séparées en ensembles d'entraînement et de test selon une répartition 80/20.


In [7]:
hist = read_table(spark, "reviews_history")

labeled = (
    hist
    .filter((F.col("rating") >= 4) | (F.col("rating") <= 2))
    .withColumn(
        "label",
        F.when(F.col("rating") >= 4, F.lit(1.0))
         .otherwise(F.lit(0.0))
    )
)

labeled.groupBy("label").count().show()

train, test = labeled.randomSplit([0.8, 0.2], seed=42)

train = train.cache()
test = test.cache()

print("Train :", train.count())
print("Test :", test.count())

+-----+------+
|label| count|
+-----+------+
|  1.0|129980|
|  0.0| 45793|
+-----+------+

Train : 140532
Test : 35241


## 4.2 Pipeline

Le pipeline transforme le texte brut en variables exploitables par le modèle : tokenisation, suppression des mots vides français et anglais, vectorisation des mots puis calcul du TF-IDF. Une régression logistique est ensuite entraînée pour prédire le sentiment des avis.

In [8]:
# Stop words français + anglais
stop_fr = StopWordsRemover.loadDefaultStopWords("french")
stop_en = StopWordsRemover.loadDefaultStopWords("english")
stop_words = list(set(stop_fr + stop_en))

def make_pipeline(classifier, tfidf=True):
    tokenizer = RegexTokenizer(
        inputCol="text",
        outputCol="words",
        pattern=r"\W+",
        toLowercase=True
    )

    remover = StopWordsRemover(
        inputCol="words",
        outputCol="filtered_words",
        stopWords=stop_words
    )

    cv = CountVectorizer(
        inputCol="filtered_words",
        outputCol="tf",
        vocabSize=10000,
        minDF=2
    )

    stages = [tokenizer, remover, cv]

    if tfidf:
        idf = IDF(
            inputCol="tf",
            outputCol="features"
        )
        stages.append(idf)
        classifier.setFeaturesCol("features")
    else:
        classifier.setFeaturesCol("tf")

    classifier.setLabelCol("label")

    stages.append(classifier)

    return Pipeline(stages=stages)

In [9]:
lr = LogisticRegression(maxIter=20)

pipeline_lr = make_pipeline(lr, tfidf=True)

start = time.time()
model_lr = pipeline_lr.fit(train)
lr_train_time = time.time() - start

print(f"Temps d'entraînement : {lr_train_time:.2f} s")

Temps d'entraînement : 12.47 s


## 4.3 Évaluation

Le modèle est évalué sur les données de test à l'aide de plusieurs métriques : accuracy, F1-score et aire sous la courbe ROC. La matrice de confusion permet d'observer les bonnes et mauvaises prédictions, tandis que l'analyse des avis mal classés aide à comprendre les limites du modèle.

In [10]:
# Prédictions sur le jeu de test
pred_lr = model_lr.transform(test)

# Accuracy
accuracy_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = accuracy_eval.evaluate(pred_lr)

# F1
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

f1 = f1_eval.evaluate(pred_lr)

# AUC ROC
auc_eval = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = auc_eval.evaluate(pred_lr)

print(f"Accuracy : {accuracy:.4f}")
print(f"F1       : {f1:.4f}")
print(f"AUC      : {auc:.4f}")

Accuracy : 0.9573
F1       : 0.9576
AUC      : 0.9641


In [11]:
pred_lr.groupBy(
    "label",
    "prediction"
).count().orderBy(
    "label",
    "prediction"
).show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0| 8573|
|  0.0|       1.0|  595|
|  1.0|       0.0|  909|
|  1.0|       1.0|25164|
+-----+----------+-----+



In [12]:
pred_lr.filter(
    F.col("label") != F.col("prediction")
).select(
    "text",
    "rating",
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)

+-------------------------------------------------------------------------------------+------+-----+----------+------------------------------------------+
|text                                                                                 |rating|label|prediction|probability                               |
+-------------------------------------------------------------------------------------+------+-----+----------+------------------------------------------+
|Livraison en retard et colis abime honnetement #sav                                  |4     |1.0  |0.0       |[0.8798634667101709,0.1201365332898291]   |
|Livraison rapide et produit conforme. excellent jean, je recommande #happy           |1     |0.0  |1.0       |[0.0010378407023688408,0.9989621592976312]|
|Ne fonctionne pas :( #retour                                                         |4     |1.0  |0.0       |[0.8630172956676541,0.13698270433234594]  |
|Tres decu par ce mascara. au top, je rachete #prix                   

## 4.4 Comparaison avec NaiveBayes

Un modèle Naive Bayes est entraîné afin de comparer ses performances avec celles de la régression logistique. Deux représentations du texte sont testées : CountVectorizer seul et TF-IDF, afin de comparer les métriques et le temps d'entraînement.


In [17]:
# NaiveBayes + CountVectorizer
nb_cv = NaiveBayes()

pipeline_nb_cv = make_pipeline(nb_cv, tfidf=False)

start = time.time()
model_nb_cv = pipeline_nb_cv.fit(train)
nb_cv_time = time.time() - start

pred_nb_cv = model_nb_cv.transform(test)

print(f"Temps NaiveBayes + CountVectorizer : {nb_cv_time:.2f} s")

Temps NaiveBayes + CountVectorizer : 2.91 s


In [18]:
#NaiveBayes + TF-IDF :
nb_tfidf = NaiveBayes()

pipeline_nb_tfidf = make_pipeline(nb_tfidf, tfidf=True)

start = time.time()
model_nb_tfidf = pipeline_nb_tfidf.fit(train)
nb_tfidf_time = time.time() - start

pred_nb_tfidf = model_nb_tfidf.transform(test)

print(f"Temps NaiveBayes + TF-IDF : {nb_tfidf_time:.2f} s")

Temps NaiveBayes + TF-IDF : 4.10 s


In [19]:
#calculer les métriques :
def evaluate_model(predictions):
    accuracy = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="accuracy"
    ).evaluate(predictions)

    f1 = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="f1"
    ).evaluate(predictions)

    auc = BinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    ).evaluate(predictions)

    return accuracy, f1, auc

In [20]:
acc_nb_cv, f1_nb_cv, auc_nb_cv = evaluate_model(pred_nb_cv)
acc_nb_tfidf, f1_nb_tfidf, auc_nb_tfidf = evaluate_model(pred_nb_tfidf)

print("NaiveBayes + CountVectorizer")
print("Accuracy :", round(acc_nb_cv, 4))
print("F1       :", round(f1_nb_cv, 4))
print("AUC      :", round(auc_nb_cv, 4))

print("\nNaiveBayes + TF-IDF")
print("Accuracy :", round(acc_nb_tfidf, 4))
print("F1       :", round(f1_nb_tfidf, 4))
print("AUC      :", round(auc_nb_tfidf, 4))

NaiveBayes + CountVectorizer
Accuracy : 0.9524
F1       : 0.9528
AUC      : 0.7032

NaiveBayes + TF-IDF
Accuracy : 0.9479
F1       : 0.9485
AUC      : 0.7575


In [22]:
#tableau comparatif
import pandas as pd

results = pd.DataFrame([
    ["LogisticRegression", "TF-IDF", accuracy, f1, auc, lr_train_time],
    ["NaiveBayes", "CountVectorizer", acc_nb_cv, f1_nb_cv, auc_nb_cv, nb_cv_time],
    ["NaiveBayes", "TF-IDF", acc_nb_tfidf, f1_nb_tfidf, auc_nb_tfidf, nb_tfidf_time]
], columns=["Modèle", "Features", "Accuracy", "F1", "AUC", "Temps (s)"])

results

,Modèle,Features,Accuracy,F1,AUC,Temps (s)
0,LogisticRegression,TF-IDF,0.957322,0.957552,0.964061,12.466675
1,NaiveBayes,CountVectorizer,0.952357,0.952836,0.703171,2.910096
2,NaiveBayes,TF-IDF,0.947930,0.948544,0.757506,4.096849


## 4.5 Sauvegarde / rechargement

Le meilleur PipelineModel est sauvegardé sur disque afin de pouvoir être réutilisé sans devoir réentraîner le modèle. Il est ensuite rechargé pour vérifier qu'il peut être appliqué directement à de nouvelles données.


In [23]:
model_path = f"{MODEL_DIR}/sentiment_lr"

model_lr.write().overwrite().save(model_path)

print("Modèle sauvegardé dans :", model_path)

Modèle sauvegardé dans : /home/jovyan/work/data/models/sentiment_lr


In [24]:
loaded_model = PipelineModel.load(model_path)

test_loaded = loaded_model.transform(test.limit(10))

test_loaded.select(
    "text",
    "rating",
    "label",
    "prediction"
).show(10, truncate=False)

+--------------------------------------------------------------------------------------------------------+------+-----+----------+
|text                                                                                                    |rating|label|prediction|
+--------------------------------------------------------------------------------------------------------+------+-----+----------+
|Service client reactif et sympa honnetement. livraison rapide et produit conforme :( #livres #recommande|5     |1.0  |1.0       |
|Rapport qualite prix top apres une semaine d'utilisation #livraison                                     |4     |1.0  |1.0       |
|Tres satisfait de ce brosse vraiment. excellent brosse, je recommande #blackfriday                      |5     |1.0  |1.0       |
|Fonctionne parfaitement apres une semaine d'utilisation :( #bonplan                                     |5     |1.0  |1.0       |
|Avoid at all costs as a gift #arnaque                                             

## 4.6 Classer les avis collectés en streaming (3 classes)

Le modèle entraîné sur l'historique est appliqué aux avis collectés en streaming. La probabilité de la classe positive est utilisée pour créer trois catégories de sentiment : positif, négatif et neutre.


In [25]:
reviews_stream = read_table(spark, "reviews_stream")

print("Nombre d'avis streamés :", reviews_stream.count())

Nombre d'avis streamés : 1276


In [26]:
pred_stream = loaded_model.transform(reviews_stream)

sentiment_df = (
    pred_stream
    .withColumn(
        "p_positive",
        vector_to_array("probability")[1]
    )
    .withColumn(
        "sentiment",
        F.when(F.col("p_positive") >= 0.65, "positive")
         .when(F.col("p_positive") <= 0.35, "negative")
         .otherwise("neutral")
    )
    .select(
        "review_id",
        "event_time",
        "category",
        "lang",
        "text",
        "rating",
        "p_positive",
        "sentiment"
    )
)

sentiment_df.show(10, truncate=False)

+------------------------------------+-----------------------+------------+----+------------------------------------------------------------------------------------------+------+--------------------+---------+
|review_id                           |event_time             |category    |lang|text                                                                                      |rating|p_positive          |sentiment|
+------------------------------------+-----------------------+------------+----+------------------------------------------------------------------------------------------+------+--------------------+---------+
|fac15dde-9051-c82f-b074-da7d158ee316|2026-09-23 15:29:32.968|mode        |fr  |Excellent jean, je recommande pour un cadeau! #top                                        |4     |0.9790852576954833  |positive |
|a254e1e7-4cc7-0e31-26c5-05b68f1b837c|2026-09-23 15:29:33.231|maison      |fr  |Tres satisfait de ce poele apres une semaine d'utilisation. super qualite #top  

In [28]:
write_table(
    sentiment_df,
    "reviews_sentiment",
    mode="overwrite"
)

print("Table reviews_sentiment enregistrée.")

Table reviews_sentiment enregistrée.


In [29]:
sentiment_df.crosstab(
    "rating",
    "sentiment"
).show()

+----------------+--------+-------+--------+
|rating_sentiment|negative|neutral|positive|
+----------------+--------+-------+--------+
|               1|     473|     15|      44|
|               5|      40|      0|    1410|
|               3|      72|    324|      48|
|               2|     274|      4|      18|
|               4|      33|      2|     911|
+----------------+--------+-------+--------+



## Réponses aux questions
- **Q4.1** : Les classes ne sont pas parfaitement équilibrées : la classe positive est plus représentée que la classe négative. Dans ce cas, l'accuracy peut être trompeuse car un modèle qui favorise la classe majoritaire peut obtenir un bon score sans bien reconnaître les deux classes. Il est donc important de regarder également le F1-score et l'AUC.
- **Q4.2** : Les avis à 3 étoiles sont majoritairement classés comme `neutral` (324 avis), mais certains sont prédits `negative` (72) ou `positive` (48). Cela s'explique notamment par le fait que les avis à 3 étoiles ont été exclus de l'entraînement. Pour améliorer le modèle, on pourrait inclure une vraie classe neutre dans les données d'entraînement et entraîner directement un modèle à trois classes, ou ajuster les seuils 0,35 / 0,65 sur un jeu de validation.

- **Q4.3** : Pour le projet Mastodon, on peut utiliser un jeu de données déjà étiqueté comme Sentiment140 pour entraîner le modèle. Le problème est que ces données sont principalement en anglais alors qu'une partie des toots peut être en français. Le vocabulaire, les expressions et les tournures françaises risquent donc d'être moins bien reconnus ; il serait préférable d'ajouter des données françaises étiquetées.